# Statistical Analysis — UCI Bank Marketing Dataset

**Author:** Naunihal Sidhu  
**Project:** Udacity — Project 2: Data and Statistical Reasoning

This notebook performs a complete statistical analysis: data ingestion, descriptive statistics, visualizations, and one hypothesis test.

## Contents
1. Environment Validation
2. Load Dataset
3. Descriptive Statistics
4. Visualizations
5. Hypothesis Test
6. Summary

## 1. Environment Validation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print('numpy', np.__version__)
print('pandas', pd.__version__)
print('scipy', stats.__name__, '/ scipy module loaded')
sns.set_theme(style='whitegrid')

numpy 2.4.4
pandas 3.0.2
scipy scipy.stats / scipy module loaded


## 2. Load Dataset

**Dataset:** UCI Bank Marketing — `bank-additional-full.csv`  
**Source:** [UCI Machine Learning Repository, ID 222](https://archive.ics.uci.edu/dataset/222/bank+marketing)  
**Reference:** Moro, S., Cortez, P., & Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing.* Decision Support Systems, 62, 22–31.

The data records direct-marketing phone campaigns by a Portuguese banking institution. Each row is one client contact; the binary target `y` records whether the client subscribed to a term deposit.

The CSV uses `;` as the field delimiter and encodes unknown categorical values as the literal string `"unknown"`.

In [2]:
import io
import os
import urllib.request
import zipfile

DATA_DIR = 'data'
DATA_PATH = os.path.join(DATA_DIR, 'bank-additional-full.csv')
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'


def ensure_dataset(path: str = DATA_PATH, url: str = SOURCE_URL) -> str:
    """Download the UCI Bank Marketing CSV from `url` if `path` is missing."""
    if os.path.exists(path):
        print(f'Using cached dataset: {path}')
        return path

    print(f'Dataset not found at {path}. Downloading from {url} ...')
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with urllib.request.urlopen(url) as resp:
        outer = zipfile.ZipFile(io.BytesIO(resp.read()))
    with outer.open('bank-additional.zip') as inner_fp:
        inner = zipfile.ZipFile(io.BytesIO(inner_fp.read()))
    target = next(n for n in inner.namelist() if n.endswith('bank-additional-full.csv'))
    with inner.open(target) as src, open(path, 'wb') as dst:
        dst.write(src.read())
    print(f'Saved dataset to {path}')
    return path


ensure_dataset()
df = pd.read_csv(DATA_PATH, sep=';')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Using cached dataset: data\bank-additional-full.csv
Shape: 41,188 rows x 21 columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
print('Column dtypes:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum())
print('\n"unknown" counts in string columns:')
print((df.select_dtypes(include='str') == 'unknown').sum().loc[lambda s: s > 0])

Column dtypes:
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

Missing values per column:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed      

## 3. Descriptive Statistics

I look at three things: numeric summaries (`df.describe`), categorical frequencies, and the shape of a few key numeric distributions. The aim is to understand the data before any test, and to flag values that may need care later (e.g. heavy skew, sentinel codes).

In [ ]:
df.describe().round(2)

**Notes on numeric summaries:**
- `age` ranges from 17 to 98 with mean ~40 — plausible adult-customer range.
- `duration` (last contact length, seconds) is highly right-skewed: mean ~258s but max 4918s. The dataset documentation warns this variable is only known *after* the call ends, so it leaks the outcome and is unsuitable as a predictor; I keep it for descriptive context only.
- `pdays` has a sentinel value `999` meaning "client was not previously contacted". The mean (~962) is therefore not a real average of days — it reflects that most clients are first-time contacts.
- `campaign` and `previous` are right-skewed counts.
- The macro-economic columns (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`) vary little within the file because the data spans a single economic period.

In [ ]:
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}\n')
for col in cat_cols:
    counts = df[col].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(2)
    print(f'--- {col} ({df[col].nunique()} unique) ---')
    print(pd.DataFrame({'count': counts, 'pct': pct}).head(12))
    print()

**Notes on categorical frequencies:**
- Target `y` is highly imbalanced: about **88.7% "no"** vs **11.3% "yes"**. Any test or model has to account for this.
- `job` is dominated by *admin.*, *blue-collar*, and *technician*.
- `education` includes *unknown* (~4.2%); `default` is almost entirely *no* with a large *unknown* slice and only 3 *yes* rows — effectively a near-constant column.
- `month` is uneven: *may* alone is ~33% of contacts, with *dec*/*mar*/*sep*/*oct* very small. This means month-based comparisons must be read carefully.
- `poutcome` is *nonexistent* for ~86% of rows (no prior campaign).

In [ ]:
print('Target balance:')
print((df['y'].value_counts(normalize=True) * 100).round(2).astype(str) + ' %')

print('\nSubscription rate by job:')
print((df.groupby('job')['y'].apply(lambda s: (s == 'yes').mean()) * 100)
      .round(2).sort_values(ascending=False).astype(str) + ' %')

key_numeric = ['age', 'duration', 'campaign', 'euribor3m']
print('\nSkewness of key numeric variables:')
print(df[key_numeric].skew().round(2))

**Notes on group differences and shape:**
- The overall subscription rate is ~11%; by `job` it ranges from below 7% (*blue-collar*, *services*) to above 25% (*student*, *retired*) — a substantial spread that motivates the hypothesis test in Section 5.
- `duration` and `campaign` are strongly right-skewed (skew >> 1), so any inference on them should prefer non-parametric or robust methods, or work on a transformed scale.
- `age` is mildly right-skewed; `euribor3m` is approximately bimodal because it tracks the policy-rate regime over the campaign period.

## 4. Visualizations

## 5. Hypothesis Test

## 6. Summary